# 💳 Credit Card Fraud Detection
**Dataset:** Kaggle Credit Card Fraud Detection  
**Goal:** Build a model to detect fraudulent transactions using EDA + ML


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap
import pickle

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('All libraries loaded ✅')

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('--- Basic Info ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum().sum(), 'missing values')
print('\n--- Class Distribution ---')
print(df['Class'].value_counts())
print(f'Fraud rate: {df["Class"].mean()*100:.3f}%')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Class Imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Transaction Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Legitimate (99.83%)', 'Fraud (0.17%)'],
            colors=['steelblue', 'crimson'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Proportion')

plt.suptitle('Severe Class Imbalance in the Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Transaction Amount: Fraud vs Legitimate
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fraud = df[df['Class'] == 1]['Amount']
legit = df[df['Class'] == 0]['Amount']

axes[0].hist(legit, bins=80, color='steelblue', alpha=0.7, label='Legitimate')
axes[0].hist(fraud, bins=80, color='crimson', alpha=0.7, label='Fraud')
axes[0].set_xlim(0, 500)
axes[0].set_title('Transaction Amount Distribution (0–$500)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

df.boxplot(column='Amount', by='Class', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='crimson', linewidth=2))
axes[1].set_title('Amount by Class')
axes[1].set_xlabel('Class (0=Legit, 1=Fraud)')
axes[1].set_ylabel('Amount ($)')
plt.suptitle('')

print(f'Avg Fraud Amount:  ${fraud.mean():.2f}')
print(f'Avg Legit Amount:  ${legit.mean():.2f}')
print(f'Max Fraud Amount:  ${fraud.max():.2f}')

plt.tight_layout()
plt.savefig('amount_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.3 Transaction Time Patterns
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

df['Hour'] = (df['Time'] // 3600) % 24

legit_hourly = df[df['Class']==0].groupby('Hour').size()
fraud_hourly = df[df['Class']==1].groupby('Hour').size()

axes[0].plot(legit_hourly.index, legit_hourly.values, color='steelblue', label='Legitimate', linewidth=2)
axes[0].set_title('Legitimate Transactions by Hour')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].bar(fraud_hourly.index, fraud_hourly.values, color='crimson', alpha=0.8, label='Fraud')
axes[1].set_title('Fraudulent Transactions by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Transaction Patterns Over Time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('time_patterns.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.4 Feature Distributions: Top PCA components
top_features = ['V1', 'V2', 'V3', 'V4', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18']

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].hist(df[df['Class']==0][feat], bins=60, color='steelblue', alpha=0.6, label='Legit', density=True)
    axes[i].hist(df[df['Class']==1][feat], bins=60, color='crimson', alpha=0.6, label='Fraud', density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=7)

plt.suptitle('PCA Feature Distributions: Fraud vs Legitimate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3.5 Correlation Heatmap (fraud transactions only)
plt.figure(figsize=(16, 10))
corr = df.drop(columns=['Time']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, annot=False, fmt='.1f')
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Preprocessing

In [ ]:
# Reload data to ensure clean state
df = pd.read_csv('../data/creditcard.csv')

# Create Hour column before scaling
df['Hour'] = (df['Time'] // 3600) % 24

# Scale Amount and Time, drop originals
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time', 'Hour'], inplace=True)

# Check for and remove NaN values
print(f'NaN values before cleaning: {df.isnull().sum().sum()}')
df = df.dropna()
print(f'NaN values after cleaning: {df.isnull().sum().sum()}')
print(f'Dataset shape after cleaning: {df.shape}')

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Fraud in train: {y_train.sum()} | Fraud in test: {y_test.sum()}')

## 5. Model Training & Evaluation

In [ ]:
# Helper: evaluate any model
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f'\n{'='*50}')
    print(f'  {model_name}')
    print(f'{'='*50}')
    print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
    print(f'ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}')
    print(f'Avg Precision: {average_precision_score(y_test, y_prob):.4f}')
    
    return y_pred, y_prob

In [ ]:
# 5.1 Baseline: Logistic Regression (no SMOTE)
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)
lr_pred, lr_prob = evaluate_model(lr_base, X_test, y_test, 'Logistic Regression (Baseline)')

In [ ]:
# 5.2 Apply SMOTE to training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Class 0: {(y_train_sm==0).sum()}, Class 1: {(y_train_sm==1).sum()}')

In [ ]:
# 5.3 Logistic Regression with SMOTE
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_sm, y_train_sm)
lr_smote_pred, lr_smote_prob = evaluate_model(lr_smote, X_test, y_test, 'Logistic Regression + SMOTE')

In [ ]:
# 5.4 Random Forest with SMOTE
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_pred, rf_prob = evaluate_model(rf, X_test, y_test, 'Random Forest + SMOTE')

In [ ]:
# 5.5 XGBoost with class_weight
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    scale_pos_weight=scale_pos,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
xgb_pred, xgb_prob = evaluate_model(xgb, X_test, y_test, 'XGBoost (scale_pos_weight)')

In [ ]:
# 5.6 ROC Curves Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = {
    'LR Baseline': lr_prob,
    'LR + SMOTE': lr_smote_prob,
    'Random Forest': rf_prob,
    'XGBoost': xgb_prob
}
colors = ['gray', 'steelblue', 'seagreen', 'crimson']

for (name, prob), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

axes[0].plot([0,1],[0,1],'k--', linewidth=1)
axes[0].set_title('ROC Curve Comparison')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=8)

for (name, prob), color in zip(models.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, linewidth=2)

axes[1].set_title('Precision-Recall Curve (more meaningful for imbalanced data)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8)

plt.suptitle('Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5.7 Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
model_preds = [
    ('LR Baseline', lr_pred),
    ('LR + SMOTE', lr_smote_pred),
    ('Random Forest', rf_pred),
    ('XGBoost', xgb_pred)
]
for ax, (name, pred) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

## 6. SHAP Explainability (XGBoost)

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test[:500])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:500], plot_type='bar', show=False)
plt.title('Top Features Driving Fraud Detection (SHAP)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test[:500], show=False)
plt.title('SHAP Summary: Feature Impact on Fraud Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', bbox_inches='tight')
plt.show()

## 7. Save Model for Streamlit App

In [ ]:
# Save the best model (XGBoost) and scaler
import os

# For Colab: create app folder in current working directory
app_dir = './app'
os.makedirs(app_dir, exist_ok=True)

with open(f'{app_dir}/model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

amount_scaler = StandardScaler()
amount_scaler.fit(pd.read_csv('../data/creditcard.csv')[['Amount']])
with open(f'{app_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(amount_scaler, f)

# Save feature names
feature_names = list(X.columns)
with open(f'{app_dir}/features.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print(f'✅ Files saved to {os.path.abspath(app_dir)}/')
print('📥 Download the app/ folder from Colab to your computer')

## 8. Key Findings & Conclusions

| Model | ROC-AUC | Avg Precision | Notes |
|---|---|---|---|
| LR Baseline | ~0.97 | ~0.70 | High FN rate |
| LR + SMOTE | ~0.97 | ~0.72 | Better recall |
| Random Forest | ~0.98 | ~0.85 | Good balance |
| **XGBoost** | **~0.98** | **~0.87** | **Best overall** |

### Key Insights:
- The dataset is **severely imbalanced** (0.17% fraud) — accuracy is a misleading metric
- **Precision-Recall AUC** is more informative than ROC-AUC for this problem
- **V14, V17, V12, V10** are the most important features for fraud detection
- Fraud transactions tend to occur more during **off-peak hours**
- Fraud amounts are **generally lower** than legitimate transactions (to avoid detection)
- **SMOTE + ensemble models** significantly improve recall without sacrificing precision
